# Notebook 04 v3: Threshold-Aware Proposed GA Model

This notebook depends on Notebook 03/03b v2 artifacts. If running in a separate Colab runtime, upload `notebook03b_outputs_v2.zip` before running this notebook.

The v3 version keeps the thesis method recognizable: frozen DistilBERT embeddings, corrected G1-G8 feature scores, GA-optimized global feature weights, and a linear binary classifier. It improves the GA by making fitness threshold-aware, controlling false positives during threshold selection, selecting feature branch scale more stably, comparing best-seed and averaged GA weights fairly, and training final Phase C across seeds 42, 7, and 123.

At the end, download the final `thesis_model_outputs_v3_YYYYMMDD_HHMMSS.zip` from the Colab sidebar.

## Dependency Check

In [9]:
# Install/import dependencies. Run this cell in Colab if packages are missing.
import sys, subprocess, importlib.util

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for import_name, pip_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("torch", "torch"),
    ("matplotlib", "matplotlib"),
    ("tqdm", "tqdm"),
]:
    ensure_package(import_name, pip_name)

print("Dependencies ready.")

All required packages available. []


## Paths and Split Configuration

In [10]:
from pathlib import Path
import json, os, random, re, time, zipfile, shutil, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

FEATURE_TEXT_COL = "message_raw"
EMBED_TEXT_COL = "model_text"
LABEL_COL = "label_id"
assert FEATURE_TEXT_COL == "message_raw"
assert EMBED_TEXT_COL == "model_text"
assert LABEL_COL == "label_id"

CONTENT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
BASE_DIR = CONTENT_DIR / "proposed_ga_v3_workspace" if CONTENT_DIR.name == "content" else Path.cwd() / "proposed_ga_v3_workspace"
BASE_DIR.mkdir(parents=True, exist_ok=True)

zip03b = CONTENT_DIR / "notebook03b_outputs_v2.zip"
zip03 = CONTENT_DIR / "notebook03_outputs_v2.zip"
if zip03b.exists():
    print(f"Found Notebook 03b v2 output ZIP, extracting into {BASE_DIR}: {zip03b}")
    with zipfile.ZipFile(zip03b, "r") as zf:
        zf.extractall(BASE_DIR)
elif zip03.exists():
    print(f"Found Notebook 03 v2 output ZIP, extracting into {BASE_DIR}: {zip03}")
    with zipfile.ZipFile(zip03, "r") as zf:
        zf.extractall(BASE_DIR)
else:
    print("No notebook03b_outputs_v2.zip or notebook03_outputs_v2.zip found in /content. Checking existing workspace files.")

ARTIFACTS_DIR = BASE_DIR / "artifacts"
RESULTS_DIR = BASE_DIR / "results"
REPORTS_DIR = BASE_DIR / "reports"
MODELS_DIR = BASE_DIR / "trained_models"
FEATURE_DIR = ARTIFACTS_DIR / "features"
EMBED_DIR = ARTIFACTS_DIR / "embeddings"
METADATA_DIR = ARTIFACTS_DIR / "metadata"
GA_DIR = ARTIFACTS_DIR / "ga_runs"
METRICS_DIR = RESULTS_DIR / "metrics"
PRED_DIR = RESULTS_DIR / "predictions"
FIGURE_DIR = RESULTS_DIR / "figures"
DEGRADATION_DIR = RESULTS_DIR / "degradation_tables"
for d in [FEATURE_DIR, EMBED_DIR, METADATA_DIR, GA_DIR, MODELS_DIR, METRICS_DIR, PRED_DIR, FIGURE_DIR, DEGRADATION_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

required_artifacts_ok = (
    FEATURE_DIR.exists() and EMBED_DIR.exists() and METADATA_DIR.exists()
    and any(FEATURE_DIR.glob("*_features_g1_g8.csv"))
    and any(EMBED_DIR.glob("*_distilbert_embeddings.npy"))
)
if not required_artifacts_ok:
    raise FileNotFoundError("Required Notebook 03 artifacts not found. Upload notebook03b_outputs_v2.zip or notebook03_outputs_v2.zip.")

feature_config = json.loads((METADATA_DIR / "feature_extraction_config.json").read_text(encoding="utf-8"))
embedding_config = json.loads((METADATA_DIR / "embedding_extraction_config.json").read_text(encoding="utf-8"))
assert feature_config["feature_text_col"] == "message_raw"
assert embedding_config["embedding_text_col"] == "model_text"

G_FEATURES = [
    "G1_URL_Signals",
    "G2_OTP_Numeric_Density",
    "G3_Obfuscation",
    "G4_Urgency_Threat_Cues",
    "G5_Action_Directives",
    "G6_Financial_Terms",
    "G7_Auth_Secrets_Request",
    "G8_Brand_Impersonation",
]
assert feature_config["feature_order"] == G_FEATURES

SPLITS = {}  # v3 can run from feature artifacts alone; raw CSVs are optional.
EVAL_SPLITS = ["test_clean", "test_adv_10", "test_adv_20", "test_adv_30"]
GA_VALIDATION_SPLITS = ["val_clean", "val_adv_10", "val_adv_20", "val_adv_30"]
SEEDS = [42, 7, 123]
for split in ["train_clean", "val_clean", *GA_VALIDATION_SPLITS, *EVAL_SPLITS]:
    f = FEATURE_DIR / f"{split}_features_g1_g8.csv"
    e = EMBED_DIR / f"{split}_distilbert_embeddings.npy"
    if not f.exists():
        raise FileNotFoundError(f"Missing feature artifact for {split}: {f}")
    if not e.exists():
        raise FileNotFoundError(f"Missing embedding artifact for {split}: {e}")

random.seed(42)
np.random.seed(42)
print("BASE_DIR =", BASE_DIR)
print("Artifacts verified. Features from message_raw; embeddings from model_text.")

Found Notebook 03b output ZIP, extracting into /content: /content/notebook03b_outputs_v2.zip
BASE_DIR = /content


## Training and Evaluation Helpers

In [11]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)

feature_config = json.loads((METADATA_DIR / "feature_extraction_config.json").read_text(encoding="utf-8"))
embedding_config = json.loads((METADATA_DIR / "embedding_extraction_config.json").read_text(encoding="utf-8"))
assert feature_config["feature_text_col"] == "message_raw"
assert feature_config["embedding_text_col"] == "model_text"
assert feature_config["feature_order"] == G_FEATURES
assert embedding_config["embedding_text_col"] == "model_text"
assert embedding_config["feature_text_col"] == "message_raw"
print("Artifact configs verified: features from message_raw; embeddings from model_text")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def load_dataset_labels(split):
    # Prefer original dataset CSVs when available, but allow separate-runtime execution
    # from Notebook 03 feature artifacts alone. Notebook 03 carries message_raw and
    # model_text forward into each corrected feature CSV for prediction inspection.
    if split in SPLITS and SPLITS[split].exists():
        df = pd.read_csv(SPLITS[split])
        if LABEL_COL not in df.columns:
            raise ValueError(f"{SPLITS[split]} is missing {LABEL_COL}")
        return df[LABEL_COL].astype(int).to_numpy(), df
    feature_path = FEATURE_DIR / f"{split}_features_g1_g8.csv"
    if not feature_path.exists():
        raise FileNotFoundError(f"Missing dataset CSV and feature artifact for {split}: {feature_path}")
    df = pd.read_csv(feature_path)
    if LABEL_COL not in df.columns:
        raise ValueError(f"{feature_path} is missing {LABEL_COL}")
    return df[LABEL_COL].astype(int).to_numpy(), df

def load_feature_frame(split):
    path = FEATURE_DIR / f"{split}_features_g1_g8.csv"
    if not path.exists(): raise FileNotFoundError(f"Missing corrected feature file: {path}. Run Notebook 03 corrected first.")
    df = pd.read_csv(path)
    missing = [c for c in G_FEATURES if c not in df.columns]
    if missing: raise ValueError(f"{path} missing columns: {missing}")
    return df

def load_embeddings(split):
    path = EMBED_DIR / f"{split}_distilbert_embeddings.npy"
    if not path.exists(): raise FileNotFoundError(f"Missing embedding file: {path}. Run Notebook 03 corrected first.")
    arr = np.load(path).astype("float32")
    if arr.ndim != 2 or arr.shape[1] != 768: raise ValueError(f"Expected 768-d embeddings for {split}, got {arr.shape}")
    return arr

def load_fused(split, feature_weights, feature_branch_scale=1.0):
    y_data, data_df = load_dataset_labels(split)
    feat_df = load_feature_frame(split)
    emb = load_embeddings(split)
    y_feat = feat_df["label_id"].astype(int).to_numpy()
    if len(data_df) != len(feat_df) or len(data_df) != len(emb): raise ValueError(f"Row count mismatch in {split}")
    if not np.array_equal(y_data, y_feat): raise ValueError(f"Label ordering mismatch for {split}")
    feats = feat_df[G_FEATURES].to_numpy(dtype="float32") * np.asarray(feature_weights, dtype="float32") * float(feature_branch_scale)
    x = np.concatenate([emb, feats], axis=1).astype("float32")
    if x.shape[1] != 776: raise ValueError(f"Expected fused dimension 776 for {split}, got {x.shape}")
    return x, y_data.astype("float32"), feat_df, data_df

class LinearHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(776, 1)
    def forward(self, x):
        return self.linear(x).squeeze(1)

def compute_metrics(y_true, y_prob, model_name, split, seed=None, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    support_ham = int((y_true == 0).sum())
    support_smishing = int((y_true == 1).sum())

    accuracy = (tp + tn) / max(len(y_true), 1)
    precision_smishing = tp / max(tp + fp, 1)
    recall_smishing = tp / max(tp + fn, 1)
    f1_smishing = (
        2 * precision_smishing * recall_smishing / max(precision_smishing + recall_smishing, 1e-12)
    )

    false_negative_rate = fn / max(fn + tp, 1)
    false_positive_rate = fp / max(fp + tn, 1)

    row = {
        "model": model_name,
        "split": split,
        "accuracy": accuracy,
        "precision_smishing": precision_smishing,
        "recall_smishing": recall_smishing,
        "f1_smishing": f1_smishing,
        "false_negative_rate": false_negative_rate,
        "false_positive_rate": false_positive_rate,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "support_ham": support_ham,
        "support_smishing": support_smishing,
        "threshold": float(threshold),
    }

    if seed is not None:
        row["seed"] = int(seed)

    return row

def train_head(model_name, feature_weights, seed=42, epochs=60, batch_size=256, lr=1e-3, patience=5, feature_branch_scale=1.0):
    set_seed(seed)
    x_train, y_train, _, _ = load_fused("train_clean", feature_weights, feature_branch_scale)
    x_val, y_val, _, _ = load_fused("val_clean", feature_weights, feature_branch_scale)
    model = LinearHead().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    loader = DataLoader(TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)), batch_size=batch_size, shuffle=True)
    best_state, best_loss, best_epoch, stale, history = None, float("inf"), 0, 0, []
    for epoch in range(1, epochs + 1):
        model.train(); total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); opt.step()
            total += loss.item() * len(xb)
        train_loss = total / len(x_train)
        model.eval()
        with torch.no_grad():
            train_logits = model(torch.from_numpy(x_train).to(DEVICE)).cpu()
            val_logits = model(torch.from_numpy(x_val).to(DEVICE)).cpu()
            val_loss = loss_fn(val_logits, torch.from_numpy(y_val)).item()
            train_prob = torch.sigmoid(train_logits).numpy()
            val_prob = torch.sigmoid(val_logits).numpy()
        tm = compute_metrics(y_train, train_prob, model_name, "train_clean", seed)
        vm = compute_metrics(y_val, val_prob, model_name, "val_clean", seed)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "train_recall_smishing": tm["recall_smishing"], "train_f1_smishing": tm["f1_smishing"], "val_recall_smishing": vm["recall_smishing"], "val_f1_smishing": vm["f1_smishing"]})
        if val_loss < best_loss - 1e-5:
            best_loss, best_epoch, stale = val_loss, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= patience: break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch, history[-1]["epoch"]

@torch.no_grad()
def predict_prob(model, x):
    model.eval()
    return torch.sigmoid(model(torch.from_numpy(x).to(DEVICE)).cpu()).numpy()

def save_confusion(m, out_path, title):
    mat = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(mat, cmap="Blues")
    ax.set_xticks([0, 1], ["Pred Ham", "Pred Smishing"]); ax.set_yticks([0, 1], ["True Ham", "True Smishing"])
    ax.set_title(title)
    for i in range(2):
        for j in range(2): ax.text(j, i, str(mat[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); fig.tight_layout(); fig.savefig(out_path, dpi=160); plt.close(fig)

def save_training_curve(history, out_path, title):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history["epoch"], history["train_loss"], label="train loss")
    ax.plot(history["epoch"], history["val_loss"], label="val loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title(title); ax.legend()
    fig.tight_layout(); fig.savefig(out_path, dpi=160); plt.close(fig)

def save_predictions(model_name, split, y, prob, feat_df, data_df, weights, seed_prefix="", feature_branch_scale=1.0, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    out = pd.DataFrame()
    for col in ["final_row_id", "source_final_row_id", "adversarial_id", "message_raw", "adv_message_raw", "model_text"]:
        if col in data_df.columns: out[col] = data_df[col]
    out["true_label"] = y.astype(int); out["predicted_label"] = pred; out["predicted_probability"] = prob
    for col in G_FEATURES: out[col] = feat_df[col].to_numpy()
    for i, col in enumerate(G_FEATURES): out[f"weighted_{col}"] = feat_df[col].to_numpy() * float(weights[i]) * float(feature_branch_scale)
    out["selected_feature_branch_scale"] = float(feature_branch_scale)
    out["selected_threshold"] = float(threshold)
    seed_match = re.search(r"seed(\d+)", seed_prefix or "")
    if seed_match:
        out["seed"] = int(seed_match.group(1))
    out.to_csv(PRED_DIR / f"{model_name}{seed_prefix}_predictions_{split}.csv", index=False)
    return out

def evaluate_model(model, model_name, weights, splits=EVAL_SPLITS, seed=42, seed_prefix="", feature_branch_scale=1.0, threshold=0.5):
    rows = []
    for split in splits:
        x, y, feat_df, data_df = load_fused(split, weights, feature_branch_scale)
        prob = predict_prob(model, x)
        m = compute_metrics(y, prob, model_name, split, seed, threshold=threshold)
        rows.append(m)
        save_predictions(model_name, split, y, prob, feat_df, data_df, weights, seed_prefix, feature_branch_scale, threshold)
        save_confusion(m, FIGURE_DIR / f"{model_name}{seed_prefix}_confusion_matrix_{split}.png", f"{model_name} {split}")
    return pd.DataFrame(rows)


Training device: cuda
Artifact configs verified: features from message_raw; embeddings from model_text


## GA Helpers

In [12]:
FEATURE_BRANCH_SCALE_CANDIDATES = [1.0, 2.0, 3.0, 5.0, 10.0]
THRESHOLD_CANDIDATES = np.round(np.arange(0.20, 0.81, 0.05), 2)
GA_SEEDS = [42, 7, 123]
GA_POPULATION_SIZE = 30
GA_GENERATIONS = 100
GA_EARLY_STOP_STAGNANT_GENERATIONS = 15
FPR_CONSTRAINT = 0.12

GA_FITNESS_FORMULA = (
    "0.30*mean_recall + 0.30*mean_f1 + 0.20*min_recall "
    "- 0.10*max_fnr - 0.05*mean_fpr - 0.05*robustness_gap "
    "- 0.05*weight_extremeness_penalty"
)
THRESHOLD_SCORE_FORMULA = (
    "0.30*mean_recall + 0.35*mean_f1 - 0.15*mean_fnr "
    "- 0.20*mean_fpr - 0.05*robustness_gap"
)


def phase_a_train():
    uniform = np.ones(len(G_FEATURES), dtype=np.float32)
    model, history, best_epoch = train_head(
        "proposed_ga_v3_phase_a",
        uniform,
        seed=42,
        epochs=60,
        batch_size=256,
        lr=1e-3,
        patience=5,
        feature_branch_scale=1.0,
    )
    out_dir = MODELS_DIR / "proposed_ga_v3"
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "feature_weights": uniform.tolist(), "feature_branch_scale": 1.0, "best_epoch": best_epoch}, out_dir / "phase_a_head.pt")
    pd.DataFrame(history).to_csv(METRICS_DIR / "proposed_phase_a_training_history.csv", index=False)
    save_training_curve(pd.DataFrame(history), FIGURE_DIR / "proposed_phase_a_training_curve.png", "Proposed GA v3 Phase A Training Curve")
    val_metrics = evaluate_model(model, "proposed_phase_a", ["val_clean"], uniform, seed=42, feature_branch_scale=1.0, threshold=0.5)
    pd.DataFrame(val_metrics).to_csv(METRICS_DIR / "proposed_phase_a_metrics.csv", index=False)
    (REPORTS_DIR / "proposed_phase_a_summary.md").write_text(
        "# Proposed GA v3 Phase A Summary\n\n"
        "Phase A trained a uniform-weight linear head on train_clean only and used val_clean only for early stopping.\n\n"
        "Because train_clean is balanced, unweighted BCEWithLogitsLoss is acceptable for Phase A/C. False-negative priority is handled in the GA fitness and threshold selection.\n",
        encoding="utf-8",
    )
    return model


def prepare_ga_arrays(feature_branch_scale):
    arrays = {}
    for split in GA_VALIDATION_SPLITS:
        emb = load_embeddings(split)
        feat_df = load_feature_frame(split)
        y = feat_df[LABEL_COL].astype(int).to_numpy()
        features = feat_df[G_FEATURES].to_numpy(dtype=np.float32)
        arrays[split] = {"emb": emb, "features": features, "y": y, "scale": feature_branch_scale}
    return arrays


def metrics_from_arrays_for_weights(model, arrays, weights, threshold):
    out = []
    for split, payload in arrays.items():
        weighted = payload["features"] * weights.reshape(1, -1) * payload["scale"]
        x = np.hstack([payload["emb"], weighted]).astype(np.float32)
        prob = predict_prob(model, x)
        out.append(compute_metrics(payload["y"], prob, "ga_validation", split, seed=None, threshold=float(threshold)))
    return pd.DataFrame(out)


def summarize_validation_metrics(metrics_df):
    recall_values = metrics_df["recall_smishing"].to_numpy(dtype=float)
    f1_values = metrics_df["f1_smishing"].to_numpy(dtype=float)
    fnr_values = metrics_df["false_negative_rate"].to_numpy(dtype=float)
    fpr_values = metrics_df["false_positive_rate"].to_numpy(dtype=float)
    clean_recall = float(metrics_df.loc[metrics_df["split"] == "val_clean", "recall_smishing"].iloc[0])
    adv_recall_min = float(metrics_df.loc[metrics_df["split"] != "val_clean", "recall_smishing"].min())
    robustness_gap = max(0.0, clean_recall - adv_recall_min)
    return {
        "mean_recall": float(np.mean(recall_values)),
        "min_recall": float(np.min(recall_values)),
        "mean_f1": float(np.mean(f1_values)),
        "mean_fnr": float(np.mean(fnr_values)),
        "max_fnr": float(np.max(fnr_values)),
        "mean_fpr": float(np.mean(fpr_values)),
        "robustness_gap": float(robustness_gap),
    }


def ga_score_from_summary(summary, weights):
    penalty = float(np.mean((weights - 1.0) ** 2))
    score = (
        0.30 * summary["mean_recall"]
        + 0.30 * summary["mean_f1"]
        + 0.20 * summary["min_recall"]
        - 0.10 * summary["max_fnr"]
        - 0.05 * summary["mean_fpr"]
        - 0.05 * summary["robustness_gap"]
        - 0.05 * penalty
    )
    return float(score), penalty


def threshold_score_from_summary(summary):
    return float(
        0.30 * summary["mean_recall"]
        + 0.35 * summary["mean_f1"]
        - 0.15 * summary["mean_fnr"]
        - 0.20 * summary["mean_fpr"]
        - 0.05 * summary["robustness_gap"]
    )


def tune_threshold_for_weights(model, arrays, weights, use_fpr_constraint=True):
    rows = []
    for threshold in THRESHOLD_CANDIDATES:
        metrics_df = metrics_from_arrays_for_weights(model, arrays, weights, threshold)
        summary = summarize_validation_metrics(metrics_df)
        score = threshold_score_from_summary(summary)
        rows.append({"threshold": float(threshold), "threshold_score": score, **summary})
    df = pd.DataFrame(rows)
    eligible = df[df["mean_fpr"] <= FPR_CONSTRAINT]
    if use_fpr_constraint and not eligible.empty:
        selected = eligible.sort_values(["threshold_score", "mean_fpr", "mean_recall"], ascending=[False, True, False]).iloc[0].to_dict()
        rule = f"best threshold_score with mean_fpr <= {FPR_CONSTRAINT}"
        satisfied = True
    else:
        selected = df.sort_values(["threshold_score", "mean_fpr", "mean_recall"], ascending=[False, True, False]).iloc[0].to_dict()
        rule = "fallback best threshold_score without FPR constraint"
        satisfied = False
    selected["selection_rule"] = rule
    selected["fpr_constraint_satisfied"] = bool(satisfied)
    return df, selected


def threshold_aware_ga_fitness(model, arrays, weights):
    best = None
    best_metrics = None
    for threshold in THRESHOLD_CANDIDATES:
        metrics_df = metrics_from_arrays_for_weights(model, arrays, weights, threshold)
        summary = summarize_validation_metrics(metrics_df)
        score, penalty = ga_score_from_summary(summary, weights)
        row = {"threshold": float(threshold), "fitness": score, "weight_extremeness_penalty": penalty, **summary}
        if best is None or row["fitness"] > best["fitness"]:
            best = row
            best_metrics = metrics_df
    return best["fitness"], best, best_metrics


def run_ga_for_scale(model, feature_branch_scale, seed):
    set_seed(seed)
    rng = np.random.default_rng(seed)
    arrays = prepare_ga_arrays(feature_branch_scale)
    population = rng.uniform(0.0, 2.0, size=(GA_POPULATION_SIZE, len(G_FEATURES))).astype(np.float32)
    best_weights = None
    best_fitness = -1e9
    stagnant = 0
    history = []
    best_components = None

    for generation in range(1, GA_GENERATIONS + 1):
        scored = []
        for individual in population:
            fitness, components, _ = threshold_aware_ga_fitness(model, arrays, individual)
            scored.append((fitness, individual.copy(), components))
        scored.sort(key=lambda x: x[0], reverse=True)
        gen_best_fitness, gen_best_weights, gen_best_components = scored[0]
        history.append({"generation": generation, "best_fitness": gen_best_fitness, **gen_best_components})
        if gen_best_fitness > best_fitness + 1e-8:
            best_fitness = float(gen_best_fitness)
            best_weights = gen_best_weights.copy()
            best_components = dict(gen_best_components)
            stagnant = 0
        else:
            stagnant += 1
        if stagnant >= GA_EARLY_STOP_STAGNANT_GENERATIONS:
            break

        elites = [w for _, w, _ in scored[: max(2, GA_POPULATION_SIZE // 5)]]
        next_pop = [elites[0].copy(), elites[1].copy()]
        while len(next_pop) < GA_POPULATION_SIZE:
            p1 = elites[rng.integers(0, len(elites))]
            p2 = elites[rng.integers(0, len(elites))]
            mask = rng.random(len(G_FEATURES)) < 0.5
            child = np.where(mask, p1, p2).astype(np.float32)
            mut_mask = rng.random(len(G_FEATURES)) < 0.20
            child[mut_mask] += rng.normal(0.0, 0.20, size=mut_mask.sum()).astype(np.float32)
            child = np.clip(child, 0.0, 2.0)
            next_pop.append(child)
        population = np.vstack(next_pop).astype(np.float32)

    history_df = pd.DataFrame(history)
    return {
        "scale": float(feature_branch_scale),
        "seed": int(seed),
        "best_weights": best_weights,
        "best_fitness": float(best_fitness),
        "history": history_df,
        "stopped_generation": int(history_df["generation"].iloc[-1]),
        "best_components": best_components,
    }

print("V3 GA helpers ready.")

## Run Phase A and Phase B GA

In [13]:
phase_a_model = phase_a_train()

all_ga_runs = []
scale_rows = []
for scale in FEATURE_BRANCH_SCALE_CANDIDATES:
    print(f"Running threshold-aware GA for feature branch scale {scale}")
    scale_results = []
    for seed in GA_SEEDS:
        result = run_ga_for_scale(phase_a_model, feature_branch_scale=scale, seed=seed)
        scale_results.append(result)
        all_ga_runs.append(result)
        result["history"].to_csv(GA_DIR / f"ga_scale{str(scale).replace('.', 'p')}_seed{seed}_fitness_history_v3.csv", index=False)
        print(f"  seed={seed} best_fitness={result['best_fitness']:.6f} stopped_generation={result['stopped_generation']}")
    fitnesses = np.array([r["best_fitness"] for r in scale_results], dtype=float)
    mean_fprs = np.array([r["best_components"].get("mean_fpr", np.nan) for r in scale_results], dtype=float)
    mean_recalls = np.array([r["best_components"].get("mean_recall", np.nan) for r in scale_results], dtype=float)
    scale_rows.append({
        "feature_branch_scale": float(scale),
        "mean_seed_fitness": float(np.mean(fitnesses)),
        "std_seed_fitness": float(np.std(fitnesses, ddof=0)),
        "best_validation_fitness": float(np.max(fitnesses)),
        "mean_fpr": float(np.nanmean(mean_fprs)),
        "mean_recall": float(np.nanmean(mean_recalls)),
        "selection_rule": "highest mean_seed_fitness, then lower std_seed_fitness, lower mean_fpr, higher best_validation_fitness",
    })

scale_search_df = pd.DataFrame(scale_rows).sort_values(
    ["mean_seed_fitness", "std_seed_fitness", "mean_fpr", "best_validation_fitness"],
    ascending=[False, True, True, False],
).reset_index(drop=True)
scale_search_df.to_csv(GA_DIR / "feature_branch_scale_search_v3.csv", index=False)
selected_feature_branch_scale = float(scale_search_df.iloc[0]["feature_branch_scale"])
print("Selected feature branch scale:", selected_feature_branch_scale)

selected_scale_runs = [r for r in all_ga_runs if float(r["scale"]) == selected_feature_branch_scale]
for r in selected_scale_runs:
    seed = r["seed"]
    best_df = pd.DataFrame([{"feature": f, "weight": float(w), "seed": seed, "feature_branch_scale": selected_feature_branch_scale} for f, w in zip(G_FEATURES, r["best_weights"])] )
    best_df.to_csv(GA_DIR / f"ga_seed{seed}_best_weights.csv", index=False)
    best_df.to_csv(GA_DIR / f"ga_seed{seed}_best_weights_v3.csv", index=False)
    r["history"].to_csv(GA_DIR / f"ga_seed{seed}_fitness_history.csv", index=False)
    r["history"].to_csv(GA_DIR / f"ga_seed{seed}_fitness_history_v3.csv", index=False)
    config = {
        "version": "v3",
        "seed": seed,
        "feature_branch_scale": selected_feature_branch_scale,
        "population_size": GA_POPULATION_SIZE,
        "generations": GA_GENERATIONS,
        "stopped_generation": r["stopped_generation"],
        "gene_space": [0, 2],
        "threshold_candidates": THRESHOLD_CANDIDATES.tolist(),
        "fitness_formula": GA_FITNESS_FORMULA,
        "validation_splits": GA_VALIDATION_SPLITS,
        "test_splits_excluded": EVAL_SPLITS,
    }
    (GA_DIR / f"ga_seed{seed}_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    (GA_DIR / f"ga_seed{seed}_config_v3.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

best_run = max(selected_scale_runs, key=lambda r: r["best_fitness"])
best_seed_weights = best_run["best_weights"].astype(np.float32)
average_seed_weights = np.mean(np.vstack([r["best_weights"] for r in selected_scale_runs]), axis=0).astype(np.float32)

arrays_for_selected_scale = prepare_ga_arrays(selected_feature_branch_scale)
weight_candidates = {
    "best_seed_weights": best_seed_weights,
    "average_seed_weights": average_seed_weights,
}
comparison_rows = []
threshold_tuning_all = []
for name, weights in weight_candidates.items():
    tune_df, selected = tune_threshold_for_weights(phase_a_model, arrays_for_selected_scale, weights, use_fpr_constraint=True)
    tune_df.insert(0, "weight_source", name)
    threshold_tuning_all.append(tune_df)
    comparison_rows.append({
        "selected_weight_source": name,
        "feature_branch_scale": selected_feature_branch_scale,
        "selected_threshold": float(selected["threshold"]),
        "validation_score": float(selected["threshold_score"]),
        "mean_recall": float(selected["mean_recall"]),
        "mean_f1": float(selected["mean_f1"]),
        "mean_fpr": float(selected["mean_fpr"]),
        "mean_fnr": float(selected["mean_fnr"]),
        "max_fnr": float(selected["max_fnr"]),
        "robustness_gap": float(selected["robustness_gap"]),
        "selection_rule": selected["selection_rule"],
        "fpr_constraint_satisfied": bool(selected["fpr_constraint_satisfied"]),
    })

threshold_tuning_df = pd.concat(threshold_tuning_all, ignore_index=True)
threshold_tuning_df.to_csv(GA_DIR / "threshold_tuning_results_v3.csv", index=False)
weight_selection_df = pd.DataFrame(comparison_rows).sort_values(
    ["validation_score", "mean_fpr", "mean_recall"], ascending=[False, True, False]
).reset_index(drop=True)
weight_selection_df.to_csv(GA_DIR / "ga_weight_selection_comparison_v3.csv", index=False)
selected_row = weight_selection_df.iloc[0].to_dict()
selected_weight_source = selected_row["selected_weight_source"]
selected_ga_weights = weight_candidates[selected_weight_source]
selected_threshold = float(selected_row["selected_threshold"])

pd.DataFrame({"feature": G_FEATURES, "weight": average_seed_weights}).to_csv(GA_DIR / "ga_average_seed_weights_v3.csv", index=False)
pd.DataFrame({"feature": G_FEATURES, "weight": best_seed_weights}).to_csv(GA_DIR / "ga_best_overall_weights_v3.csv", index=False)
pd.DataFrame({"feature": G_FEATURES, "weight": selected_ga_weights, "selected_weight_source": selected_weight_source}).to_csv(GA_DIR / "ga_selected_final_weights_v3.csv", index=False)

stability_df = pd.DataFrame(
    np.vstack([r["best_weights"] for r in selected_scale_runs]),
    columns=G_FEATURES,
)
stability_df.insert(0, "seed", [r["seed"] for r in selected_scale_runs])
stability_summary = stability_df[G_FEATURES].agg(["mean", "std", "min", "max"]).T.reset_index().rename(columns={"index": "feature"})
stability_summary.to_csv(GA_DIR / "ga_weight_stability_summary_v3.csv", index=False)

selected_threshold_info = {
    "selected_threshold": selected_threshold,
    "selected_weight_source": selected_weight_source,
    "feature_branch_scale": selected_feature_branch_scale,
    "selection_rule": selected_row["selection_rule"],
    "fpr_constraint_satisfied": bool(selected_row["fpr_constraint_satisfied"]),
    "mean_recall": float(selected_row["mean_recall"]),
    "mean_f1": float(selected_row["mean_f1"]),
    "mean_fnr": float(selected_row["mean_fnr"]),
    "mean_fpr": float(selected_row["mean_fpr"]),
    "max_fnr": float(selected_row["max_fnr"]),
    "robustness_gap": float(selected_row["robustness_gap"]),
    "threshold_score_formula": THRESHOLD_SCORE_FORMULA,
    "ga_fitness_formula": GA_FITNESS_FORMULA,
}
(GA_DIR / "selected_threshold_v3.json").write_text(json.dumps(selected_threshold_info, indent=2), encoding="utf-8")

plt.figure(figsize=(9, 4))
plt.bar(G_FEATURES, selected_ga_weights)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Selected GA weight")
plt.title("Proposed GA v3 Selected Feature Weights")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "ga_best_weights_bar_chart_v3.png", dpi=180)
plt.close()

print("Selected weight source:", selected_weight_source)
print("Selected threshold:", selected_threshold)
print("Selected feature branch scale:", selected_feature_branch_scale)

GA scale 1.0 seed 42:   0%|          | 0/100 [00:00<?, ?it/s]

GA scale 1.0 seed 7:   0%|          | 0/100 [00:00<?, ?it/s]

GA scale 1.0 seed 123:   0%|          | 0/100 [00:00<?, ?it/s]

GA scale 2.0 seed 42:   0%|          | 0/100 [00:00<?, ?it/s]

GA scale 2.0 seed 7:   0%|          | 0/100 [00:00<?, ?it/s]

...

GA scale 10.0 seed 42:   0%|          | 0/100 [00:00<?, ?it/s]

GA scale 10.0 seed 7:   0%|          | 0/100 [00:00<?, ?it/s]

GA scale 10.0 seed 123:   0%|          | 0/100 [00:00<?, ?it/s]

Selected feature branch scale: 5.0


,feature,selected_weight
0,G1_URL_Signals,0.903179
1,G2_OTP_Numeric_Density,0.778802
2,G3_Obfuscation,1.379225
3,G4_Urgency_Threat_Cues,0.887191
4,G5_Action_Directives,0.916878
5,G6_Financial_Terms,1.011678
6,G7_Auth_Secrets_Request,1.017170
7,G8_Brand_Impersonation,0.666921


## Run Phase C Final Proposed Model

In [19]:
proposed_dir = MODELS_DIR / "proposed_ga_v3"
proposed_dir.mkdir(parents=True, exist_ok=True)

all_metric_rows = []
all_histories = []
representative_predictions = {}
phase_c_threshold_tuning_frames = []
phase_c_threshold_selections = []

for seed in SEEDS:
    print(f"Training final Phase C proposed GA v3 head for seed {seed}")
    model, history, best_epoch = train_head(
        "proposed_ga_v3",
        selected_ga_weights,
        seed=seed,
        epochs=60,
        batch_size=256,
        lr=1e-3,
        patience=5,
        feature_branch_scale=selected_feature_branch_scale,
    )
    final_tune_df, final_selected_threshold = tune_threshold_for_weights(
        model,
        arrays_for_selected_scale,
        selected_ga_weights,
        use_fpr_constraint=True,
    )
    final_tune_df.insert(0, "seed", seed)
    phase_c_threshold_tuning_frames.append(final_tune_df)
    threshold_for_seed = float(final_selected_threshold["threshold"])
    phase_c_threshold_selections.append({"seed": seed, **final_selected_threshold})

    seed_dir = proposed_dir / f"seed{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    torch.save({
        "state_dict": model.state_dict(),
        "feature_weights": selected_ga_weights.tolist(),
        "selected_weight_source": selected_weight_source,
        "feature_branch_scale": selected_feature_branch_scale,
        "threshold": threshold_for_seed,
        "best_epoch": best_epoch,
        "seed": seed,
    }, seed_dir / "phase_c_final_head.pt")

    hist_df = pd.DataFrame(history)
    hist_df.insert(0, "seed", seed)
    hist_df.to_csv(METRICS_DIR / f"proposed_ga_v3_seed{seed}_training_history.csv", index=False)
    all_histories.append(hist_df)

    for split in EVAL_SPLITS:
        x, y, feat_df, data_df = load_fused(split, selected_ga_weights, feature_branch_scale=selected_feature_branch_scale)
        prob = predict_prob(model, x)
        metrics = compute_metrics(y, prob, "proposed_ga_v3", split, seed=seed, threshold=threshold_for_seed)
        metrics["selected_weight_source"] = selected_weight_source
        metrics["feature_branch_scale"] = selected_feature_branch_scale
        all_metric_rows.append(metrics)

        pred_df = save_predictions(
            "proposed_ga_v3",
            split,
            y,
            prob,
            feat_df,
            data_df,
            selected_ga_weights,
            seed_prefix=f"_seed{seed}",
            feature_branch_scale=selected_feature_branch_scale,
            threshold=threshold_for_seed,
        )
        save_confusion(metrics, FIGURE_DIR / f"proposed_ga_v3_confusion_matrix_seed{seed}_{split}.png", f"Proposed GA v3 seed {seed} {split}")
        if seed == 42:
            representative_predictions[split] = pred_df.copy()

metrics_by_seed_df = pd.DataFrame(all_metric_rows)
metrics_by_seed_df.to_csv(METRICS_DIR / "proposed_ga_v3_metrics_by_seed.csv", index=False)

metric_cols = [
    "accuracy", "precision_smishing", "recall_smishing", "f1_smishing",
    "false_negative_rate", "false_positive_rate", "tp", "tn", "fp", "fn",
    "support_ham", "support_smishing", "threshold",
]
mean_rows = []
for split, grp in metrics_by_seed_df.groupby("split"):
    row = {"model": "proposed_ga_v3", "split": split, "num_seeds": grp["seed"].nunique()}
    for col in metric_cols:
        if col in grp.columns:
            row[f"{col}_mean"] = float(grp[col].mean())
            row[f"{col}_std"] = float(grp[col].std(ddof=0))
    mean_rows.append(row)
metrics_mean_std_df = pd.DataFrame(mean_rows)
metrics_mean_std_df.to_csv(METRICS_DIR / "proposed_ga_v3_metrics_mean_std.csv", index=False)

all_histories_df = pd.concat(all_histories, ignore_index=True)
all_histories_df.to_csv(METRICS_DIR / "proposed_phase_c_training_history.csv", index=False)
for seed in SEEDS:
    one = all_histories_df[all_histories_df["seed"] == seed].copy()
    save_training_curve(one, FIGURE_DIR / f"proposed_phase_c_training_curve_seed{seed}.png", f"Proposed GA v3 Phase C Training Curve seed {seed}")

# Representative inspection files use seed 42 to avoid choosing a representative from test performance.
for split, pred_df in representative_predictions.items():
    pred_df[pred_df["true_label"].eq(1) & pred_df["predicted_label"].eq(0)].to_csv(PRED_DIR / f"proposed_ga_v3_false_negatives_{split}.csv", index=False)
    pred_df[pred_df["true_label"].eq(0) & pred_df["predicted_label"].eq(1)].to_csv(PRED_DIR / f"proposed_ga_v3_false_positives_{split}.csv", index=False)

phase_c_threshold_tuning_df = pd.concat(phase_c_threshold_tuning_frames, ignore_index=True)
phase_c_threshold_tuning_df.to_csv(GA_DIR / "phase_c_threshold_tuning_results_v3.csv", index=False)
pd.DataFrame(phase_c_threshold_selections).to_csv(GA_DIR / "phase_c_selected_thresholds_v3.csv", index=False)

selected_threshold_record = json.loads((GA_DIR / "selected_threshold_v3.json").read_text(encoding="utf-8"))
selected_threshold_record["ga_weight_selection_threshold"] = selected_threshold_record.get("selected_threshold")
selected_threshold_record["phase_c_seed_thresholds"] = [
    {"seed": int(row["seed"]), "selected_threshold": float(row["threshold"]), "fpr_constraint_satisfied": bool(row["fpr_constraint_satisfied"]), "mean_fpr": float(row["mean_fpr"]), "mean_recall": float(row["mean_recall"]), "mean_f1": float(row["mean_f1"]), "mean_fnr": float(row["mean_fnr"]), "robustness_gap": float(row["robustness_gap"])}
    for row in phase_c_threshold_selections
]
selected_threshold_record["selection_rule"] = "Phase C thresholds are tuned per final seed on validation splits only using the FPR-aware rule."
(GA_DIR / "selected_threshold_v3.json").write_text(json.dumps(selected_threshold_record, indent=2), encoding="utf-8")

print("Phase C complete. Metrics saved for seeds:", SEEDS)

,model,split,accuracy,precision_smishing,recall_smishing,f1_smishing,false_negative_rate,false_positive_rate,tp,tn,fp,fn,support_ham,support_smishing,threshold,seed
0,proposed_ga_v2,test_clean,0.927307,0.888506,0.977244,0.930765,0.022756,0.12263,773,694,97,18,791,791,0.25,42
1,proposed_ga_v2,test_adv_10,0.923515,0.887731,0.969659,0.926888,0.030341,0.12263,767,694,97,24,791,791,0.25,42
2,proposed_ga_v2,test_adv_20,0.912137,0.885343,0.946903,0.915089,0.053097,0.12263,749,694,97,42,791,791,0.25,42
3,proposed_ga_v2,test_adv_30,0.905815,0.883971,0.934260,0.908420,0.065740,0.12263,739,694,97,52,791,791,0.25,42


## Diagnostics, Degradation Tables, and Final Comparison

In [15]:
def degradation_table_from_seed_metrics(metrics_df):
    rows = []
    for seed, grp in metrics_df.groupby("seed"):
        by_split = {r["split"]: r for _, r in grp.iterrows()}
        clean = by_split.get("test_clean")
        adv30 = by_split.get("test_adv_30")
        if clean is None or adv30 is None:
            continue
        row = {"model": "proposed_ga_v3", "seed": int(seed)}
        for label, split_name in [("clean", "test_clean"), ("adv10", "test_adv_10"), ("adv20", "test_adv_20"), ("adv30", "test_adv_30")]:
            r = by_split.get(split_name)
            if r is not None:
                row[f"{label}_accuracy"] = float(r["accuracy"])
                row[f"{label}_recall"] = float(r["recall_smishing"])
                row[f"{label}_f1"] = float(r["f1_smishing"])
                row[f"{label}_fnr"] = float(r["false_negative_rate"])
                row[f"{label}_fpr"] = float(r["false_positive_rate"])
        row["accuracy_drop_clean_to_adv30"] = row.get("clean_accuracy", np.nan) - row.get("adv30_accuracy", np.nan)
        row["recall_drop"] = row.get("clean_recall", np.nan) - row.get("adv30_recall", np.nan)
        row["f1_drop"] = row.get("clean_f1", np.nan) - row.get("adv30_f1", np.nan)
        row["fnr_increase"] = row.get("adv30_fnr", np.nan) - row.get("clean_fnr", np.nan)
        rows.append(row)
    seed_df = pd.DataFrame(rows)
    mean = seed_df.drop(columns=["seed"], errors="ignore").select_dtypes(include=[np.number]).mean().to_dict()
    mean_row = {"model": "proposed_ga_v3", "seed": "mean", **mean}
    return pd.concat([seed_df, pd.DataFrame([mean_row])], ignore_index=True)

proposed_degradation_df = degradation_table_from_seed_metrics(metrics_by_seed_df)
proposed_degradation_df.to_csv(DEGRADATION_DIR / "proposed_ga_v3_degradation_table.csv", index=False)

scale_choice = scale_search_df.iloc[0].to_dict()
mean_metrics_summary = metrics_mean_std_df.to_string(index=False)
degradation_summary = proposed_degradation_df.to_string(index=False)
weight_comparison_summary = weight_selection_df.to_string(index=False)
scale_search_summary = scale_search_df.to_string(index=False)
stability_summary_text = stability_summary.to_string(index=False)
phase_c_threshold_summary = pd.DataFrame(phase_c_threshold_selections).to_string(index=False)

def maybe_read_metrics(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return None

ablation_c = maybe_read_metrics(METRICS_DIR / "ablation_c_metrics.csv")
ablation_d = maybe_read_metrics(METRICS_DIR / "ablation_d_metrics_mean_std.csv")
v2_metrics = maybe_read_metrics(METRICS_DIR / "proposed_ga_v2_metrics.csv")

def comparison_note_against(df, label):
    if df is None:
        return f"{label}: metrics unavailable."
    notes = []
    for split in EVAL_SPLITS:
        v3_row = metrics_by_seed_df[metrics_by_seed_df["split"].eq(split)]
        other_row = df[df["split"].eq(split)] if "split" in df.columns else pd.DataFrame()
        if v3_row.empty or other_row.empty:
            continue
        v3_fn = float(v3_row["fn"].mean())
        r = other_row.iloc[0]
        other_fn = float(r["fn_mean"] if "fn_mean" in r else r["fn"] if "fn" in r else np.nan)
        if np.isfinite(other_fn):
            direction = "reduced" if v3_fn < other_fn else "did not reduce" if v3_fn > other_fn else "matched"
            notes.append(f"{split}: v3 mean FN {v3_fn:.2f} vs {label} FN {other_fn:.2f}; v3 {direction} false negatives.")
    return "\n".join(notes) if notes else f"{label}: comparable FN columns unavailable."

ablation_c_note = comparison_note_against(ablation_c, "Ablation C")
ablation_d_note = comparison_note_against(ablation_d, "Ablation D")
v2_note = comparison_note_against(v2_metrics, "Proposed GA v2")

(REPORTS_DIR / "proposed_ga_v3_data_routing_check.md").write_text(f"""# Proposed GA v3 Data Routing Check

- Engineered features were required to come from `message_raw`: {feature_config['feature_text_col'] == 'message_raw'}.
- DistilBERT embeddings were required to come from `model_text`: {embedding_config['embedding_text_col'] == 'model_text'}.
- G1-G8 order was checked exactly: {G_FEATURES}.
- Fusion dimension remained 768 + 8 = 776.
- Phase A training used `train_clean`; early stopping used `val_clean`.
- GA validation used only: {GA_VALIDATION_SPLITS}.
- Phase C training used `train_clean`; early stopping used `val_clean`.
- Threshold tuning used validation splits only: {GA_VALIDATION_SPLITS}.
- Final evaluation used only: {EVAL_SPLITS}.
- Test sets were excluded from GA, threshold tuning, early stopping, and model selection.
- `val_adv_30` was treated as validation-only and was not used for final test reporting.
""", encoding="utf-8")

(REPORTS_DIR / "proposed_ga_v3_overfitting_check.md").write_text(f"""# Proposed GA v3 Overfitting Check

The final model was trained across seeds {SEEDS}. The report should be interpreted using the mean and standard deviation across seeds rather than one lucky run.

## Training Histories

Training histories are saved under `results/metrics/proposed_ga_v3_seed*_training_history.csv` and training curves under `results/figures/proposed_phase_c_training_curve_seed*.png`.

## Validation-Only Selection

Feature branch scale, GA weights, and threshold were selected using validation splits only. Test splits were not used during selection.

## Scale Search

```
{scale_search_summary}
```

## Weight Selection

```
{weight_comparison_summary}
```

## Test Mean/Std Summary

```
{mean_metrics_summary}
```

Large gaps between validation-selected behavior and test behavior should be treated as possible validation overfitting. Clean-to-adv30 changes are summarized in the degradation table.
""", encoding="utf-8")

(REPORTS_DIR / "proposed_ga_v3_robustness_improvement_summary.md").write_text(f"""# Proposed GA v3 Robustness Improvement Summary

V3 keeps the same core thesis architecture but improves the optimization procedure.

- GA fitness became threshold-aware using: `{GA_FITNESS_FORMULA}`.
- Threshold tuning used: `{THRESHOLD_SCORE_FORMULA}`.
- Preferred threshold rule: choose the best score with mean FPR <= {FPR_CONSTRAINT}; otherwise use the unconstrained best score.
- GA weight-selection threshold: {selected_threshold}.
- Final Phase C per-seed thresholds were tuned after final training on validation splits only.

```
{phase_c_threshold_summary}
```

- GA-stage FPR constraint satisfied: {selected_threshold_info['fpr_constraint_satisfied']}.
- Selected feature branch scale: {selected_feature_branch_scale}.
- Scale selection rule: highest mean seed fitness, then lower std seed fitness, lower mean FPR, higher best validation fitness.
- Selected weights came from: {selected_weight_source}.

The target is not necessarily to beat a fully fine-tuned DistilBERT model trained with adversarial augmentation. The Proposed GA target is to improve interpretability, reduce false negatives versus C/D when possible, control false positives more defensibly than v2 threshold behavior, and maintain low clean-to-adversarial degradation.
""", encoding="utf-8")

(REPORTS_DIR / "proposed_ga_v3_audit_summary.md").write_text(f"""# Proposed GA v3 Audit Summary

## Source Columns

- Features extracted from `message_raw`: {feature_config['feature_text_col'] == 'message_raw'}.
- Embeddings extracted from `model_text`: {embedding_config['embedding_text_col'] == 'model_text'}.

## Dataset Use

- Phase A: train_clean only, val_clean for early stopping.
- Phase B GA: {GA_VALIDATION_SPLITS} only.
- Phase C: train_clean only, val_clean for early stopping.
- Threshold tuning: {GA_VALIDATION_SPLITS} only.
- Test evaluation: {EVAL_SPLITS} only.

## Selection Decisions

- Feature branch scale selected: {selected_feature_branch_scale}.
- Threshold selected: {selected_threshold}.
- Threshold FPR constraint satisfied: {selected_threshold_info['fpr_constraint_satisfied']}.
- Weight source selected: {selected_weight_source}.

## Weight Stability

```
{stability_summary_text}
```

## Degradation Summary

```
{degradation_summary}
```

## Ablation Availability

- Ablation C metrics found: {ablation_c is not None}.
- Ablation D metrics found: {ablation_d is not None}.
- Proposed GA v2 metrics found: {v2_metrics is not None}.

## False Negative Comparison Notes

{ablation_c_note}

{ablation_d_note}

{v2_note}

Manual review should inspect false negatives, false positives, and whether any feature weight dominates too strongly.
""", encoding="utf-8")

(REPORTS_DIR / "proposed_ga_v3_final_summary.md").write_text(f"""# Proposed GA v3 Final Summary

Proposed GA v3 trained final linear heads across seeds {SEEDS}. Mean and standard deviation metrics are saved in `results/metrics/proposed_ga_v3_metrics_mean_std.csv`.

## Mean/Std Metrics

```
{mean_metrics_summary}
```

## Degradation

```
{degradation_summary}
```

The GA-stage selected threshold was {selected_threshold}. Final Phase C used per-seed thresholds tuned after final training on validation splits only. The selected feature branch scale was {selected_feature_branch_scale}. The selected global G1-G8 weights came from `{selected_weight_source}`.

## Comparison Notes

{ablation_c_note}

{ablation_d_note}

{v2_note}
""", encoding="utf-8")

# Optional final comparison with available metrics. Missing files are reported, not treated as failure.
comparison_rows = []
missing_models = []

def add_long_metrics(model_label, path, split_col="split"):
    if not path.exists():
        missing_models.append(model_label)
        return
    df = pd.read_csv(path)
    if "model" not in df.columns:
        df["model"] = model_label
    if split_col not in df.columns:
        missing_models.append(model_label)
        return
    for split in EVAL_SPLITS:
        row_df = df[df[split_col].eq(split)]
        if row_df.empty:
            continue
        r = row_df.iloc[0].to_dict()
        out = {"model": model_label, "split": split}
        for src, dst in [("accuracy", "accuracy"), ("accuracy_mean", "accuracy"), ("recall_smishing", "recall"), ("recall_smishing_mean", "recall"), ("f1_smishing", "f1"), ("f1_smishing_mean", "f1"), ("false_negative_rate", "FNR"), ("false_negative_rate_mean", "FNR"), ("false_positive_rate", "FPR"), ("false_positive_rate_mean", "FPR")]:
            if src in r and dst not in out:
                out[dst] = r[src]
        comparison_rows.append(out)

add_long_metrics("TF-IDF Baseline 1", METRICS_DIR / "tfidf_baseline1_metrics_mean_std.csv")
add_long_metrics("TF-IDF Ablation A", METRICS_DIR / "tfidf_ablation_a_metrics_mean_std.csv")
add_long_metrics("DistilBERT Baseline 2", METRICS_DIR / "distilbert_baseline2_metrics_mean_std.csv")
add_long_metrics("Ablation B", METRICS_DIR / "distilbert_ablation_b_metrics_mean_std.csv")
add_long_metrics("Ablation C", METRICS_DIR / "ablation_c_metrics.csv")
add_long_metrics("Ablation D", METRICS_DIR / "ablation_d_metrics_mean_std.csv")
add_long_metrics("Proposed GA v2", METRICS_DIR / "proposed_ga_v2_metrics.csv")
add_long_metrics("Proposed GA v3", METRICS_DIR / "proposed_ga_v3_metrics_mean_std.csv")

if comparison_rows:
    comp = pd.DataFrame(comparison_rows)
    wide = comp.pivot_table(index="model", columns="split", values=["accuracy", "recall", "f1", "FNR", "FPR"], aggfunc="first")
    wide.columns = [f"{split}_{metric}" for metric, split in wide.columns]
    wide = wide.reset_index()
    if "test_clean_recall" in wide.columns and "test_adv_30_recall" in wide.columns:
        wide["clean_to_adv30_recall_drop"] = wide["test_clean_recall"] - wide["test_adv_30_recall"]
    if "test_clean_f1" in wide.columns and "test_adv_30_f1" in wide.columns:
        wide["clean_to_adv30_f1_drop"] = wide["test_clean_f1"] - wide["test_adv_30_f1"]
    if "test_clean_FNR" in wide.columns and "test_adv_30_FNR" in wide.columns:
        wide["clean_to_adv30_fnr_increase"] = wide["test_adv_30_FNR"] - wide["test_clean_FNR"]
    wide.to_csv(METRICS_DIR / "final_model_comparison_all_available_v3.csv", index=False)
    comparison_text = wide.to_string(index=False)
else:
    comparison_text = "No comparison metrics were available."

(REPORTS_DIR / "final_model_comparison_all_available_v3.md").write_text(
    "# Final Model Comparison All Available v3\n\n"
    f"Missing or unavailable model files: {missing_models}\n\n"
    "```\n" + comparison_text + "\n```\n",
    encoding="utf-8",
)

print("Diagnostics, degradation tables, and reports saved.")

,model,split,seed,n_rows,accuracy,precision_smishing,recall_smishing,f1_smishing,false_negative_rate,false_positive_rate,tp,tn,fp,fn,support_ham,support_smishing,display_name,threshold
0,ablation_c,test_clean,42.0,1582.0,0.943110,0.941992,0.944374,0.943182,0.055626,0.058154,747.000000,745.000000,46.000000,44.000000,791.0,791.0,Ablation C,NaN
1,ablation_c,test_adv_10,42.0,1582.0,0.930468,0.940492,0.919090,0.929668,0.080910,0.058154,727.000000,745.000000,46.000000,64.000000,791.0,791.0,Ablation C,NaN
2,ablation_c,test_adv_20,42.0,1582.0,0.924147,0.939712,0.906448,0.922780,0.093552,0.058154,717.000000,745.000000,46.000000,74.000000,791.0,791.0,Ablation C,NaN
3,ablation_c,test_adv_30,42.0,1582.0,0.906448,0.937415,0.871049,0.903014,0.128951,0.058154,689.000000,745.000000,46.000000,102.000000,791.0,791.0,Ablation C,NaN
4,NaN,test_adv_10,NaN,NaN,0.934682,0.945963,0.922040,0.933833,0.077960,0.052676,729.333333,749.333333,41.666667,61.666667,791.0,791.0,Ablation D,NaN
5,NaN,test_adv_20,NaN,NaN,0.923515,0.944694,0.899705,0.921644,0.100295,0.052676,711.666667,749.333333,41.666667,79.333333,791.0,791.0,Ablation D,NaN
6,NaN,test_adv_30,NaN,NaN,0.908555,0.942901,0.869785,0.904861,0.130215,0.052676,688.000000,749.333333,41.666667,103.000000,791.0,791.0,Ablation D,NaN
7,NaN,test_clean,NaN,NaN,0.942056,0.946765,0.936789,0.941750,0.063211,0.052676,741.000000,749.333333,41.666667,50.000000,791.0,791.0,Ablation D,NaN
8,proposed_ga_v2,test_clean,42.0,NaN,0.919090,0.870391,0.984829,0.924081,0.015171,0.146650,779.000000,675.000000,116.000000,12.000000,791.0,791.0,Proposed GA v2,0.2
9,proposed_ga_v2,test_adv_10,42.0,NaN,0.914665,0.869369,0.975980,0.919595,0.024020,0.146650,772.000000,675.000000,116.000000,19.000000,791.0,791.0,Proposed GA v2,0.2


## Create Final Download ZIP

In [16]:
folders_to_zip = ["artifacts", "trained_models", "results", "reports"]
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
bundle_dir = CONTENT_DIR / f"thesis_model_outputs_v3_{timestamp}"
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

for folder_name in folders_to_zip:
    src = BASE_DIR / folder_name
    dst = bundle_dir / folder_name
    if src.exists():
        print(f"Adding: {src}")
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        print(f"Skipping missing folder: {src}")

zip_base = CONTENT_DIR / f"thesis_model_outputs_v3_{timestamp}"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=bundle_dir)
print("Final thesis output ZIP created.")
print(zip_path)
print("Download this ZIP from the Colab sidebar before closing or resetting the runtime.")

Adding: /content/artifacts
Adding: /content/trained_models
Adding: /content/results
Adding: /content/reports
? Final thesis v2 output ZIP created.
Download this ZIP from the Colab sidebar before closing or resetting the runtime.
/content/thesis_model_outputs_v2_20260517_172049.zip


## Final ZIP Checklist

In [17]:
required_zip_entries = [
    "artifacts/features",
    "artifacts/embeddings",
    "artifacts/metadata",
    "artifacts/ga_runs",
    "trained_models/proposed_ga_v3",
    "results/metrics",
    "results/predictions",
    "results/figures",
    "results/degradation_tables",
    "reports",
]
with zipfile.ZipFile(zip_path, "r") as zf:
    names = set(zf.namelist())
    for required in required_zip_entries:
        prefix = required.rstrip("/") + "/"
        present = any(name.startswith(prefix) for name in names)
        print(f"{required}: {'FOUND' if present else 'MISSING'}")
print("ZIP checklist complete:", zip_path)

artifacts/features: FOUND
artifacts/embeddings: FOUND
artifacts/metadata: FOUND
artifacts/ga_runs: FOUND
trained_models: FOUND
trained_models/proposed_ga_v2: FOUND
results/metrics: FOUND
results/predictions: FOUND
results/degradation_tables: FOUND
results/figures: FOUND
reports: FOUND
ZIP checklist complete: /content/thesis_model_outputs_v2_20260517_172049.zip
